In [1]:

import re
from urllib.parse import urlparse

KEYWORDS = ["login", "verify", "secure", "update", "account", "bank"]


BRANDS = {
    "paypal": "paypal.com",
    "google": "google.com",
    "microsoft": "microsoftonline.com",
    "apple": "apple.com",
    "amazon": "amazon.com",
    "facebook": "facebook.com",
    "netflix": "netflix.com",
}


def phish_score(url):
    """Score a URL 0-100 based on common phishing indicators."""
    p = urlparse(url)
    score = 0
    reasons = []

 
    if not url.startswith("https"):
        score += 30
        reasons.append("No HTTPS (unencrypted connection)")

  
    for kw in KEYWORDS:
        if kw in p.netloc.lower():
            score += 20
            reasons.append(f"Suspicious keyword in domain: '{kw}'")
            break

    
    for brand, real_domain in BRANDS.items():
        if brand in p.netloc.lower() and real_domain not in p.netloc.lower():
            score += 35
            reasons.append(f"Impersonates brand '{brand}' (not on {real_domain})")
            break

   
    if p.netloc.count(".") > 3:
        score += 25
        reasons.append("Excessive subdomains (possible subdomain abuse)")

   
    if re.search(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", p.netloc):
        score += 40
        reasons.append("Uses raw IP address instead of domain name")

    return min(score, 100), reasons


def classify(score):
    """Turn a numeric score into a human-readable risk label."""
    if score >= 70:
        return "HIGH RISK"
    elif score >= 40:
        return "MEDIUM RISK"
    elif score > 0:
        return "LOW RISK"
    else:
        return "LIKELY SAFE"


def main():
  
    urls = [
        "https://paypal-login.evil.com/verify",
        "https://github.com",
        "http://192.168.1.1/login/account/verify",
        "https://secure.update.account.paypal.evil-domain.com",
        "https://www.google.com",
        "http://bank-secure-login.com/update",
        "https://accounts.google.com/signin",
        "http://verify.account.bank.suspicious-site.ru",
        "https://www.wikipedia.org",
        "https://login.microsoftonline.com",
    ]

    print(f"{'URL':<55} {'Score':<8} {'Risk':<12}")
    print("-" * 80)

    for u in urls:
        score, reasons = phish_score(u)
        risk = classify(score)
        print(f"{u:<55} {score:<8} {risk:<12}")
        if reasons:
            for r in reasons:
                print(f"    - {r}")
        print()


if __name__ == "__main__":
    main()

URL                                                     Score    Risk        
--------------------------------------------------------------------------------
https://paypal-login.evil.com/verify                    55       MEDIUM RISK 
    - Suspicious keyword in domain: 'login'
    - Impersonates brand 'paypal' (not on paypal.com)

https://github.com                                      0        LIKELY SAFE 

http://192.168.1.1/login/account/verify                 70       HIGH RISK   
    - No HTTPS (unencrypted connection)
    - Uses raw IP address instead of domain name

https://secure.update.account.paypal.evil-domain.com    80       HIGH RISK   
    - Suspicious keyword in domain: 'secure'
    - Impersonates brand 'paypal' (not on paypal.com)
    - Excessive subdomains (possible subdomain abuse)

https://www.google.com                                  0        LIKELY SAFE 

http://bank-secure-login.com/update                     50       MEDIUM RISK 
    - No HTTPS (unencrypted 